# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the FAIR² dataset using the `mlcroissant` library, following best practices for data science and reproducible ML. All entities (e.g., record sets, fields, columns) are referenced by their `@id` identifiers for clarity and traceability.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

This tabular dataset contains 77 records of cancer survivors with second primary colorectal cancer, encompassing clinical, pathological, and biomarker variables.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running on a new environment)
# !pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and establish a handle for efficient record access.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's discover available record sets, their fields, and `@id`s defined in the dataset. This will help in selecting data for further analysis.

We print all `@id` values for record sets and fields.

In [ ]:
# List all record set @ids and their fields
record_sets = list(dataset.record_sets)
if not record_sets:
    # Fallback: Try to guess possible record set IDs from the dataset's internal JSON-LD
    jsonld = dataset.json_ld
    record_sets = []
    for e in jsonld.get("@graph", []) if "@graph" in jsonld else []:
        types = e.get("@type", [])
        if isinstance(types, str):
            types = [types]
        if "cr:RecordSet" in types or ("RecordSet" in types):
            record_sets.append(e["@id"])
    if not record_sets:
        # Last-resort: Look for a record set in possible top-level properties
        record_sets = metadata.record_set if hasattr(metadata, "record_set") else []

print("Available record set @ids:")
for rid in record_sets:
    print(f"  - {rid}")

# For each record set, print its fields and columns
for rid in record_sets:
    rs = dataset.record_set(rid)
    print(f"\nRecord set @id: {rid}")
    print("  Field @ids:")
    for field in rs.fields:
        print(f"    - {field['@id']} (label: {field.get('rdfs:label', field.get('label', 'N/A'))})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Column @ids:")
        for col in rs.columns:
            print(f"    - {col['@id']} (label: {col.get('rdfs:label', col.get('label', 'N/A'))})")

## 3. Data Extraction

Now we'll load data from a specific record set using its `@id` and convert it to a pandas DataFrame. For this dataset, the main record set typically containing clinical data is used. All field/column references are made by their `@id`s.

_**Note:** If there are multiple record sets (tables), feel free to extract more than one._

In [ ]:
# Define the list of record set @ids to extract
# (Update these based on the actual record set @ids found above; here we use a sample placeholder @id)
# If the output of the previous cell lists no record sets, consult the dataset documentation for correct @id.
record_sets_to_extract = record_sets if record_sets else []  # Use all found record sets, or set manually
# Uncomment and edit the next line if record_sets is empty and you know a specific @id
# record_sets_to_extract = ["your_record_set_@id_here"]

dataframes = {}
for rs_id in record_sets_to_extract:
    # Records yields dicts with field @id as keys
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for record set @id: {rs_id}")
        print(f"Columns (field @ids): {dataframes[rs_id].columns.tolist()}")
        display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process the data, using column `@id`s, for common tasks such as filtering by a clinical variable, normalizing a numeric attribute, and grouping by categorical variables. Replace placeholders with detected numeric and groupable field @ids as appropriate.

In [ ]:
# Select which record set to perform EDA on
if dataframes:
    record_set_id = list(dataframes.keys())[0]  # Take the first loaded record set for demo
    df = dataframes[record_set_id]
    print(f"Columns available in DataFrame for record set @id {record_set_id}:\n", list(df.columns))

    # CHOOSE A NUMERIC FIELD BY @id (e.g., for 'age' or 'interval_months')
    possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in {'i', 'f'}]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
    else:
        print("No obvious numeric field detected; set 'numeric_field' to known column.")
        numeric_field = None

    # Apply a filter if possible
    threshold = 60 if numeric_field and df[numeric_field].max() > 60 else 10  # Adapt threshold
    if numeric_field:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (total: {len(filtered_df)}):")
        display(filtered_df.head())
    else:
        filtered_df = df
    # Normalize
    if numeric_field:
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

    # Choose a field to group by (e.g., sex, tumor_location, etc.)
    possible_group_fields = [col for col in df.columns if df[col].dtype == object or df[col].dtype.name == 'category']
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (averages shown):")
        display(grouped_df.head())

## 5. Visualization

Visualize some distributions or relationships in the dataset using matplotlib and seaborn, referencing all fields/columns by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library, making all references to record sets and fields by their Croissant `@id` for reproducibility. You can further adapt these patterns for advanced modeling, joined analyses, or exporting subsets using the `mlcroissant` metadata-driven approach.